# The recommender, tested once

*Text shortened on 23 September; code and outputs are unchanged. The plan as committed before running is in commit `aff5f3d`.*

Notebook 13 chose the methods and settings on development data, so its numbers are
optimistic. This scores them once on the 2025 test half: 1,032 line-ups (3 July to 5 October)
never used for recommending. Their win results were seen in notebook 12, which asked a
different question. **Committed to git before it ran.**

**Decided in advance:**

| | |
|---|---|
| Headline | Gradient boosting, 45-day half-life (`random_state=0`) |
| Also reported | Classifier + map × agent (21 days), co-occurrence (21), plain classifier (45) |
| The bar | The naive rule: most-played agents on this map lately (7 days) |
| History | 2024 plus 2025 up to Masters Toronto, then week by week, each week using only earlier games |
| Measures | Top-1 and top-3 (hide one agent, give the other four), and from scratch |
| Margin of error | 5,000 resamples of whole matches, seed 0 |
| Verdict | "Beats the naive rule" only if the bottom of the top-3 lead's 95% range is above zero |
| Breakdowns | One only: Corrode, a brand-new map, against every other map |

Expected: lower than development.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 200)

from dataset import build_map_table, split_season
from recommend import Classifier, CoOccurrence, Popularity, lineup_table, summarise, walk_forward

SEED, RESAMPLES = 0, 5000
HEADLINE, BAR = "gradient boosting", "popularity (naive rule)"
METHODS = {
    "gradient boosting": lambda: Classifier(45, estimator=HistGradientBoostingClassifier(random_state=0)),
    "classifier + map x agent": lambda: Classifier(21, interactions=True),
    "co-occurrence": lambda: CoOccurrence(21),
    "classifier": lambda: Classifier(45),
    "popularity (naive rule)": lambda: Popularity(7),
}

## The line-ups

In [2]:
learn_2025, test_2025 = split_season(build_map_table())
history = pd.concat([lineup_table(build_map_table("vct_2024")), lineup_table(learn_2025)], ignore_index=True)
to_score = lineup_table(test_2025)

print(f"history : {len(history):,} line-ups, {history.played_at.min():%d %b %Y} to {history.played_at.max():%d %b %Y}")
print(f"scored  : {len(to_score):,} line-ups from {to_score.match_id.nunique()} matches, "
      f"{to_score.played_at.min():%d %b} to {to_score.played_at.max():%d %b %Y}, "
      f"over {to_score.played_at.dt.to_period('W').nunique()} weeks")
print(f"          x 5 hidden agents = {len(to_score) * 5:,} complete-a-line-up questions")

new_maps = sorted(set(to_score["map"]) - set(history["map"]))
if new_maps:
    on_new_map = to_score["map"].isin(new_maps)
    print(f"\nmaps with no history at all: {new_maps}, first played "
          f"{to_score.loc[on_new_map, 'played_at'].min():%d %b}, {on_new_map.sum()} line-ups")
else:
    print("\nevery map scored here has history")

history : 3,720 line-ups, 17 Feb 2024 to 22 Jun 2025
scored  : 1,032 line-ups from 202 matches, 03 Jul to 05 Oct 2025, over 13 weeks
          x 5 hidden agents = 5,160 complete-a-line-up questions

maps with no history at all: ['Corrode'], first played 03 Jul, 140 line-ups


## Run every method once

Gradient boosting refits every week, so this takes several minutes.

In [3]:
runs = {name: walk_forward(history, to_score, make) for name, make in METHODS.items()}

## The result

In [4]:
results = pd.DataFrame({name: summarise(r) for name, r in runs.items()}).T
print(f"{len(to_score):,} test line-ups.  Picking at random from the ~23 agents left gets top-3 about 13%.\n")
print(results.to_string(formatters={"top-1": "{:.1%}".format, "top-3": "{:.1%}".format,
                                    "from scratch, of 5": "{:.2f}".format}))

1,032 test line-ups.  Picking at random from the ~23 agents left gets top-3 about 13%.

                         top-1 top-3 from scratch, of 5
gradient boosting        74.8% 91.3%               3.31
classifier + map x agent 75.1% 91.4%               3.38
co-occurrence            76.4% 91.2%               3.40
classifier               71.2% 89.1%               3.38
popularity (naive rule)  56.0% 77.3%               3.44


## The margin of error

In [5]:
def per_lineup(result, cutoff):
    return (np.vstack(result["ranks"].to_numpy()) <= cutoff).mean(axis=1)

match_of = runs[HEADLINE]["match_id"].to_numpy()
match_ids = np.unique(match_of)
rows_of = {m: np.flatnonzero(match_of == m) for m in match_ids}
rng = np.random.default_rng(SEED)
draws = [np.concatenate([rows_of[m] for m in rng.choice(match_ids, len(match_ids))]) for _ in range(RESAMPLES)]

def interval(values):
    return np.percentile([values[d].mean() for d in draws], [2.5, 97.5])

for label, cutoff in [("top-3", 3), ("top-1", 1)]:
    v = per_lineup(runs[HEADLINE], cutoff)
    lo, hi = interval(v)
    print(f"{HEADLINE} {label}: {v.mean():.1%}   95% range {lo:.1%} to {hi:.1%}")

lead = per_lineup(runs[HEADLINE], 3) - per_lineup(runs[BAR], 3)
lead_lo, lead_hi = interval(lead)
print(f"\ntop-3 lead over the naive rule: {lead.mean() * 100:+.1f} points   "
      f"95% range {lead_lo * 100:+.1f} to {lead_hi * 100:+.1f}")

if lead.mean() > 0 and lead_lo > 0:
    verdict = "BEATS THE NAIVE RULE"
elif lead.mean() > 0:
    verdict = "AHEAD, BUT NOT DISTINGUISHABLE FROM THE NAIVE RULE"
else:
    verdict = "DOES NOT BEAT THE NAIVE RULE"
print(f"\nby the rule decided in advance: {verdict}")

gradient boosting top-3: 91.3%   95% range 90.0% to 92.6%
gradient boosting top-1: 74.8%   95% range 73.0% to 76.7%

top-3 lead over the naive rule: +14.0 points   95% range +12.6 to +15.5

by the rule decided in advance: BEATS THE NAIVE RULE


## The one pre-registered breakdown: a brand-new map

In [6]:
headline_run = runs[HEADLINE]
on_new = headline_run["map"].isin(new_maps)
groups = [(f"{', '.join(new_maps)} (new)", on_new), ("every other map", ~on_new)] if new_maps else []
if not groups:
    print("no brand-new map in the weeks scored -- nothing to break down")
for label, mask in groups:
    part = headline_run[mask]
    print(f"{label:18} {len(part):4} line-ups   " +
          "   ".join(f"{k} {v:.1%}" if k != "from scratch, of 5" else f"{k} {v:.2f}"
                     for k, v in summarise(part).items()))

Corrode (new)       140 line-ups   top-1 57.0%   top-3 80.1%   from scratch, of 5 2.98
every other map     892 line-ups   top-1 77.6%   top-3 93.0%   from scratch, of 5 3.37


## What this means

*Written after the cells above ran, in a separate commit.*

**91.3% top-3 (range 90.0% to 92.6%) and 74.8% named first**, on 1,032 unseen line-ups. **By the
rule set in advance, it beats the naive rule:** a lead of 14.0 points (12.6 to 15.5).

- **Higher than development (88.0%), but not luck:** the naive rule rose more (71.1% → 77.3%),
  so this half was easier. The lead shrank a little, from about 17 points to 14.
- **The top three methods finish within about a point** (91.3%, 91.4%, 91.2%). Gradient boosting
  stays the headline by the rule.
- **From the map alone, the naive rule is still best** (3.44 of five against 3.31).
- **Corrode, a brand-new map, costs about 13 points** (80.1% against 93.0%).

It measures imitation, not winning.